# encoder
# decoder
# train
# generate

In [2]:
import torch
import torch.nn as nn

In [3]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.0)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 要限定 input的shape
        # input.shape (batch_size, seqlen)
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        output, hidden = self.rnn(input)
        return output, hidden


In [5]:
import random
def make_sample(sample_size):
    ep = sample_size
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample(100)
print(inputs[0:10])
print(targets[0:10])

print("len(inputs)== len(targets)",len(inputs)== len(targets))

['2017-3-6', '2043-5-3', '2028-4-9', '2016-12-18', '1976-6-14', '2012-7-28', '1981-8-23', '2033-2-12', '2025-10-7', '2036-10-4']
['6/3/2017', '3/5/2043', '9/4/2028', '18/12/2016', '14/6/1976', '28/7/2012', '23/8/1981', '12/2/2033', '7/10/2025', '4/10/2036']
len(inputs)== len(targets) True


In [6]:
def tensor_from_string(s, add_eos=False):
    input_tensor = torch.tensor([stoi[ch] for ch in s], dtype=torch.long)
    if add_eos:
        new_value = torch.tensor([EOS_token])
        input_tensor = torch.cat([input_tensor, new_value],dim=0)
    return input_tensor.unsqueeze(0)

def string_from_tensor(tensor):
    s = ''
    tensor = tensor.squeeze(0)
    for token in tensor:
        idx = token.item()
        if idx != EOS_token:
            s += itos[idx]
    return s

In [7]:

x = tensor_from_string("2002-1-23")
print(x, x.shape)
y = tensor_from_string("23/1/2002", add_eos=True)
print(y, y.shape)


tensor([[ 4,  2,  2,  4, 12,  3, 12,  4,  5]]) torch.Size([1, 9])
tensor([[ 4,  5, 13,  3, 13,  4,  2,  2,  4,  1]]) torch.Size([1, 10])


In [8]:
print(string_from_tensor(x))
print(string_from_tensor(y))

2002-1-23
23/1/2002


In [9]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

x torch.Size([1, 9])
encoder_output: torch.Size([1, 9, 5])
encoder_hidden: torch.Size([1, 1, 5])


In [10]:
from torch import Tensor
from typing import Optional


class DecoderRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, output_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_hidden: Tensor, target_tensor: Optional[Tensor] = None, max_len: int = 20):
        # encoder_output是encoder输出的 hiddeng向量
        # target_tensor.shape is (batch_size, seq_len)
        # foward.shape is (batch_size, seq_len,vocab_size)
        batch_size = encoder_hidden.shape[1]
        hidden = encoder_hidden
        input_token = SOS_token 
        outputs = []
        # input_token的格式是 (batch_size,seq_len)
        input_token = torch.full((batch_size, 1), SOS_token, dtype=torch.long) #设置起始词元
        loop_len = max_len
        if target_tensor is not None:
            loop_len = target_tensor.shape[1]

        for i in range(loop_len):
            logits, hidden, predictIdx = self.forward_step(hidden, input_token) 
            outputs.append(logits)

            if target_tensor is not None:
                input_token = target_tensor[:,i].unsqueeze(1)
            else :
                input_token = predictIdx.detach()
        decoder_outputs = torch.stack(tensors=outputs, dim=1)
        return decoder_outputs, hidden
    
    def forward_step(self, hidden, input_token):
        # input_token.shape是（batch_size,seq_len)
        embedded = self.embedding(input_token)
        # rnn的hidden输入是(batch_size,seq_len,hidden)
        output, hidden = self.rnn(embedded, hidden)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        logits = self.out(hidden[-1]) # hidden[-1]表示最后一个num
        predictIdx = logits.argmax(dim = -1,keepdim= True)
        return logits, hidden, predictIdx

    

In [11]:
input_token = torch.full((1, 1), SOS_token, dtype=torch.long)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)
logits, decoder_hidden, predictIdx = decoder.forward_step(encoder_hidden, input_token)

print("decoder_hidden:", decoder_hidden.shape)
print("logits:", logits.shape)
print("predictIdx", predictIdx)


decoder_hidden: torch.Size([1, 1, 5])
logits: torch.Size([1, 14])
predictIdx tensor([[6]])


In [12]:
print(decoder.out.out_features)

14


In [13]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

x = tensor_from_string("2002-1-23")
y = tensor_from_string("23/1/2002", add_eos=True)

encoder_output, encoder_hidden = encoder(x)
decoder_outputs, decoder_hidden = decoder(encoder_hidden, y)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)


decoder_outputs: torch.Size([1, 10, 14])
decoder_hidden: torch.Size([1, 1, 5])
target: torch.Size([1, 10])


In [14]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

x = tensor_from_string("2002-1-23")
y = tensor_from_string("23/1/2002", add_eos=True)

encoder_output, encoder_hidden = encoder(x)
decoder_outputs, decoder_hidden = decoder(encoder_hidden, y)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)

decoder_outputs: torch.Size([1, 10, 14])
decoder_hidden: torch.Size([1, 1, 5])
target: torch.Size([1, 10])


In [15]:
from torch.nn.modules.loss import CrossEntropyLoss


def train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    input_tensor = tensor_from_string(input_str) 
    target_tensor = tensor_from_string(target_str, True)
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    encoder_output, encoder_hidden = encoder(input_tensor)
    decoder_outputs, decoder_hidden = decoder(encoder_hidden, target_tensor)
    logits = decoder_outputs.reshape(-1, vocab_size)
    target = target_tensor.reshape(-1)
    loss = criterion(logits, target)
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item()


In [16]:
hidden_size = 32
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=0.01)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=0.01)

criterion = nn.CrossEntropyLoss()

loss = train_one_sample(
    "2002-1-23",
    "23/1/2002",
    encoder,
    decoder,
    encoder_optimizer,
    decoder_optimizer,
    criterion,
)

print(loss)

2.7400169372558594


In [17]:
def generate(input_str, encoder, decoder, max_len=20):
    with torch.no_grad():
        input_tensor = tensor_from_string(input_str)
        encoder_output, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden = decoder(encoder_hidden, None, max_len)
        # decoder_outputs.shape (batch,seq_len,vocab_size)
        pred_tokens = decoder_outputs.argmax(dim=2)
        outputs = []
        for token in pred_tokens[0]:
            if token.item() == EOS_token:
                break
            outputs.append(token)
        if len(outputs) == 0:
            return ""
        
    
    return string_from_tensor(torch.stack(outputs).unsqueeze(0))

In [ ]:
fixed_samples = [
    ("2002-1-23", "23/1/2002"),
    ("1999-12-8", "8/12/1999"),
    ("2020-3-7", "7/3/2020"),
    ("1950-11-28", "28/11/1950"),
    ("2048-6-15", "15/6/2048"),
]

def eval_samples(samples, encoder, decoder, verbose=False):
    encoder.eval()
    decoder.eval()

    correct_count = 0

    for input_str, target_str in samples:
        pred = generate(input_str, encoder, decoder)
        correct = pred == target_str

        if correct:
            correct_count += 1

        if verbose:
            print("input:  ", input_str)
            print("pred:   ", pred)
            print("target: ", target_str)
            print("correct:", correct)
            print("---")

    total = len(samples)
    acc = correct_count / total

    return correct_count, total, acc


In [ ]:
import random
# 循环训练
epoch = 1000
sample_size = 100
loss_group = sample_size//10
lr = 0.001
inputs, targets = make_sample(sample_size)
samples = list(zip(inputs, targets))
# samples = fixed_samples
hidden_size = 256

encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size,vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr)

criterion = nn.CrossEntropyLoss()
for j in range(epoch):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    random.shuffle(samples)
    for i, (input_str, target_str) in enumerate(samples):
        loss = train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += loss
    
    print("epoch:",j+1, ", avg_epoch_loss:",epoch_loss/len(samples))

    
            
train_correct, train_acc = eval_samples(encoder, decoder, samples)
print("train accuracy:", train_correct, "/", len(samples), train_acc)

test_inputs, test_targets = make_sample(100)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_acc = eval_samples( encoder, decoder, test_samples)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)

epoch: 1 , avg_epoch_loss: 1.4372233903408052
epoch: 2 , avg_epoch_loss: 0.9620035088062286
epoch: 3 , avg_epoch_loss: 0.8192708343267441
epoch: 4 , avg_epoch_loss: 0.757887973189354
epoch: 5 , avg_epoch_loss: 0.7258491870760918
epoch: 6 , avg_epoch_loss: 0.7024237361550331
epoch: 7 , avg_epoch_loss: 0.6694003629684449
epoch: 8 , avg_epoch_loss: 0.6870170697569847
epoch: 9 , avg_epoch_loss: 0.6598763123154641
epoch: 10 , avg_epoch_loss: 0.6319284492731094
epoch: 11 , avg_epoch_loss: 0.6153895044326783
epoch: 12 , avg_epoch_loss: 0.6179552564024925
epoch: 13 , avg_epoch_loss: 0.6201445665955544
epoch: 14 , avg_epoch_loss: 0.6250352185964584
epoch: 15 , avg_epoch_loss: 0.5908946481347084
epoch: 16 , avg_epoch_loss: 0.5595824933052063
epoch: 17 , avg_epoch_loss: 0.5464529943466186
epoch: 18 , avg_epoch_loss: 0.5356620714068413
epoch: 19 , avg_epoch_loss: 0.5162962341308593
epoch: 20 , avg_epoch_loss: 0.5215399892628193
epoch: 21 , avg_epoch_loss: 0.5149478113651276
epoch: 22 , avg_epoch_l

AttributeError: 'list' object has no attribute 'eval'

In [33]:
test_inputs, test_targets = make_sample(100)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_acc = eval_samples( encoder, decoder, test_samples)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)

test accuracy: 0 / 100 0.0


In [34]:
train_correct, train_acc = eval_samples(encoder, decoder, samples)
print("train accuracy:", train_correct, "/", len(samples), train_acc)

train accuracy: 87 / 100 17.4
